In [15]:
import tensorflow as tf
import matplotlib.pyplot as plt
from models import build_generator, build_discriminator
from dataset import get_dataset
import os

os.makedirs("results", exist_ok=True)

# ============================================================
# 0️⃣ Hyperparameters
# ============================================================
lambda_cyc_start = 5.0
lambda_cyc_end   = 5.0

lambda_id_start  = 2.5
lambda_id_end    = 2.5

alpha = 0.02  # weight for perceptual cycle loss

img_size = 256
batch_size = 4
EPOCHS = 100

# ============================================================
# 1️⃣ Build Generators and Discriminators
# ============================================================
# G: X -> Y (animal -> origami)
# F: Y -> X (origami -> animal)
G = build_generator(image_size=img_size, n_blocks=9)
F = build_generator(image_size=img_size, n_blocks=9)
D_X = build_discriminator(image_size=img_size)  # discrim on X domain
D_Y = build_discriminator(image_size=img_size)  # discrim on Y domain

# ============================================================
# 2️⃣ Optimizers and Loss Functions
# ============================================================
G_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
F_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
D_X_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
D_Y_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)

mse = tf.keras.losses.MeanSquaredError()
L1  = lambda a, b: tf.reduce_mean(tf.abs(a - b))

# ============================================================
# 2.5️⃣ VGG19 for Perceptual Feature Extraction
# ============================================================
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input

# Pretrained VGG19 on ImageNet, used as a fixed feature extractor
vgg = VGG19(include_top=False, weights="imagenet",
            input_shape=(img_size, img_size, 3))
vgg.trainable = False

# Use a mid-level conv layer as perceptual features
vgg_feature_layer = "block3_conv3"
vgg_feature_extractor = tf.keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer(vgg_feature_layer).output
)

def perceptual_loss(img1, img2):
    """
    Perceptual L1 loss between VGG19 feature maps.
    img1, img2 are expected in [-1, 1] range, RGB.
    """
    # [-1, 1] -> [0, 255]
    img1 = (img1 + 1.0) * 127.5
    img2 = (img2 + 1.0) * 127.5

    # VGG19 preprocessing (BGR ordering, mean subtraction, etc.)
    img1 = preprocess_input(img1)
    img2 = preprocess_input(img2)

    # Extract feature maps
    feat1 = vgg_feature_extractor(img1)
    feat2 = vgg_feature_extractor(img2)

    # L1 in feature space
    return tf.reduce_mean(tf.abs(feat1 - feat2))


# ============================================================
# 3️⃣ One Training Step (Eager mode, no @tf.function)
# ============================================================
@tf.function
def train_step(real_x, real_y, lambda_cyc, lambda_id, step_count):
    with tf.GradientTape(persistent=True) as tape:
        # -------- Forward --------
        fake_y = G(real_x, training=True)
        fake_x = F(real_y, training=True)
        
        # -------- Cycle --------
        cyc_x = F(fake_y, training=True)
        cyc_y = G(fake_x, training=True)
        
        # -------- Identity --------
        same_x = F(real_x, training=True)
        same_y = G(real_y, training=True)
        
        # -------- Discriminators --------
        D_X_real = D_X(real_x, training=True)
        D_X_fake = D_X(fake_x, training=True)
        D_Y_real = D_Y(real_y, training=True)
        D_Y_fake = D_Y(fake_y, training=True)
        
        # -------- LSGAN Losses --------
        G_GAN_loss = mse(tf.ones_like(D_Y_fake), D_Y_fake)
        F_GAN_loss = mse(tf.ones_like(D_X_fake), D_X_fake)
        
        D_X_loss = 0.5 * (mse(tf.ones_like(D_X_real), D_X_real) +
                          mse(tf.zeros_like(D_X_fake), D_X_fake))
        D_Y_loss = 0.5 * (mse(tf.ones_like(D_Y_real), D_Y_real) +
                          mse(tf.zeros_like(D_Y_fake), D_Y_fake))
        
        cycle_G = L1(cyc_y, real_y) + alpha * perceptual_loss(cyc_y, real_y)
        cycle_F = L1(cyc_x, real_x) + alpha * perceptual_loss(cyc_x, real_x)
        
        # -------- Identity Loss --------
        id_G = L1(same_y, real_y)
        id_F = L1(same_x, real_x)
        
        # -------- Final Losses --------
        G_loss = G_GAN_loss + lambda_cyc * cycle_G + lambda_id * id_G
        F_loss = F_GAN_loss + lambda_cyc * cycle_F + lambda_id * id_F
    
    # ---- Apply Gradients ----
    G_grads = tape.gradient(G_loss, G.trainable_variables)
    F_grads = tape.gradient(F_loss, F.trainable_variables)
    D_X_grads = tape.gradient(D_X_loss, D_X.trainable_variables)
    D_Y_grads = tape.gradient(D_Y_loss, D_Y.trainable_variables)
    del tape
    
    G_optimizer.apply_gradients(zip(G_grads, G.trainable_variables))
    F_optimizer.apply_gradients(zip(F_grads, F.trainable_variables))
    D_X_optimizer.apply_gradients(zip(D_X_grads, D_X.trainable_variables))
    D_Y_optimizer.apply_gradients(zip(D_Y_grads, D_Y.trainable_variables))
    
    return G_loss, F_loss, D_X_loss, D_Y_loss



# ============================================================
# 4️⃣ Dataset
# ============================================================
trainX = get_dataset('data/butterfly_real/*.jpeg',
                     batch_size=batch_size, img_size=img_size)
trainY = get_dataset('data/butterfly_origami/*.jpg',
                     batch_size=batch_size, img_size=img_size)

# ============================================================
# 5️⃣ Training Loop with λ decay
# ============================================================
step_count = tf.Variable(0, dtype=tf.int32, trainable=False)

for epoch in range(EPOCHS):
    lambda_cyc = 10.0
    lambda_id = 5.0
    
    for real_x, real_y in zip(trainX, trainY):
        G_loss, F_loss, DX_loss, DY_loss = train_step(
            real_x, real_y,
            tf.constant(lambda_cyc, dtype=tf.float32),
            tf.constant(lambda_id, dtype=tf.float32),
            step_count
        )
        step_count.assign_add(1)  # 步数+1
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1} | G={float(G_loss):.3f} F={float(F_loss):.3f} "
              f"DX={float(DX_loss):.3f} DY={float(DY_loss):.3f}")
        # Take one batch sample from X and visualize translation
        sample = next(iter(trainX))
        fake_y = G(sample, training=False)

        plt.figure(figsize=(6, 3))
        plt.subplot(1, 2, 1)
        plt.imshow(((sample[0] + 1) / 2.0).numpy())
        plt.title("Real X")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(((fake_y[0] + 1) / 2.0).numpy())
        plt.title("Fake Y (origami style)")
        plt.axis("off")

        plt.tight_layout()
        plt.savefig(f"results/epoch_{epoch+1}.png")
        plt.close()

print("Training complete ✅")


KeyboardInterrupt: 